In [18]:
import os
import librosa
import numpy as np
import soundfile as sf
import numpy as np

In [12]:
data_dir = "./GTZAN/Data/genres_original"
corrupt_files = []

for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith(".wav"):
            file_path = os.path.join(root, file)
            try:
                # Attempt to load the 30-second clip
                y, sr = librosa.load(file_path, duration=30)
            except Exception as e:
                print(f"[ERROR] Corrupt file detected: {file_path}")
                print(f"\t\tException: {e}")
                print(f"\t\t{type(e)}")
                corrupt_files.append(file_path)

print(f"\nVerification complete. Found {len(corrupt_files)} unreadable/corrupt files.")

[ERROR] Corrupt file detected: ./GTZAN/Data/genres_original\jazz\jazz.00054.wav
		Exception: Error opening './GTZAN/Data/genres_original\\jazz\\jazz.00054.wav': Format not recognised.
		<class 'soundfile.LibsndfileError'>

Verification complete. Found 1 unreadable/corrupt files.


In [21]:
EXPECTED_GENRES = {
    "blues", "classical", "country", "disco", "hiphop", 
    "jazz", "metal", "pop", "reggae", "rock"
}
EXPECTED_SR = 22050
EXPECTED_CHANNELS = 1

s_lb = 29.9
s_ub = 30.1
# Acceptable frame count tolerance bounds (29.5 to 30.5 seconds)
EXPECTED_MIN_FRAMES = int(EXPECTED_SR * s_lb)  # 651,475 frames
EXPECTED_MAX_FRAMES = int(EXPECTED_SR * s_ub)  # 671,525 frames

actual_min_frame = EXPECTED_MAX_FRAMES
actual_max_frame = EXPECTED_MIN_FRAMES

issues = {"corrupt": [], "wrong_specs": [], "structural": [], "silent": []}

# 1. Structural Check: Verify folders exist
subdirs = {d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))}
if not EXPECTED_GENRES.issubset(subdirs):
    issues["structural"].append(f"Missing one or more expected genre folders.")

for genre in EXPECTED_GENRES:
    genre_path = os.path.join(data_dir, genre)
    if not os.path.exists(genre_path):
        continue
    
    files = [f for f in os.listdir(genre_path) if f.endswith(".wav")]
    if len(files) != 100:
        issues["structural"].append(f"Folder '{genre}' contains {len(files)} files instead of 100.")

    for file in files:
        file_path = os.path.join(genre_path, file)
        
        try:
            # 2. Fast metadata inspection using soundfile
            info = sf.info(file_path)
            
            # 3. Check sample rate and channels
            if info.samplerate != EXPECTED_SR or info.channels != EXPECTED_CHANNELS:
                issues["wrong_specs"].append(f"{file_path} -> SR: {info.samplerate}, Channels: {info.channels}")
            
            # 4. Check frame count against tolerance window
            if not (EXPECTED_MIN_FRAMES <= info.frames <= EXPECTED_MAX_FRAMES):
                issues["wrong_specs"].append(f"{file_path} -> Out-of-bounds frame count: {info.frames}")
                if info.frames < actual_min_frame: actual_min_frame = info.frames
                if info.frames > actual_max_frame: actual_max_frame = info.frames
                    

            # 5. Load full audio via librosa to check for silent/zero-padded files
            y, sr = librosa.load(file_path, sr=None)
            if np.all(y == 0) or np.max(np.abs(y)) < 1e-5:
                issues["silent"].append(file_path)

        except Exception as e:
            # Catches truly corrupt/unreadable files like jazz.00054.wav
            issues["corrupt"].append((file_path, str(e)))
            print(f'Exception: {e}')

# --- Summary Report ---
print("\n" + "="*40)
print("       GTZAN VERIFICATION SUMMARY")
print("="*40)

total_issues = sum(len(v) for v in issues.values())

print(
    f'sampling bounds at SR = {EXPECTED_SR}',
    f'\n\tseconds : [{s_lb}, {s_ub}]',
    f'\n\tframes : [{EXPECTED_MIN_FRAMES}, {EXPECTED_MAX_FRAMES}]\n'
)

print(f'actual max : {actual_max_frame}')
print(f'actual min : {actual_min_frame}')

for category, items in issues.items():
    print(f"\n{category.capitalize()} issues found: {len(items)}")
    # Print up to the first 5 examples for each category if any exist
    for item in items[:5]:
        print(f"  - {item}")
    if len(items) > 5:
        print(f"  ... and {len(items) - 5} more.")

print("\n" + "="*50)
if total_issues == 0:
    print("Result: Dataset is pristine and fully valid!")
else:
    print(f"Result: Found {total_issues} total items requiring attention.")

Exception: Error opening './GTZAN/Data/genres_original\\jazz\\jazz.00054.wav': Format not recognised.

       GTZAN VERIFICATION SUMMARY
sampling bounds at SR = 22050 
	seconds : [29.9, 30.1] 
	frames : [659295, 663705]

actual max : 675808
actual min : 663705

Corrupt issues found: 1
  - ('./GTZAN/Data/genres_original\\jazz\\jazz.00054.wav', "Error opening './GTZAN/Data/genres_original\\\\jazz\\\\jazz.00054.wav': Format not recognised.")

Wrong_specs issues found: 40
  - ./GTZAN/Data/genres_original\disco\disco.00000.wav -> Out-of-bounds frame count: 664180
  - ./GTZAN/Data/genres_original\disco\disco.00001.wav -> Out-of-bounds frame count: 666160
  - ./GTZAN/Data/genres_original\disco\disco.00003.wav -> Out-of-bounds frame count: 668140
  - ./GTZAN/Data/genres_original\disco\disco.00005.wav -> Out-of-bounds frame count: 667920
  - ./GTZAN/Data/genres_original\disco\disco.00008.wav -> Out-of-bounds frame count: 665060
  ... and 35 more.

Structural issues found: 0

Silent issues found